In [1]:
import pandas as pd
from tqdm.auto import tqdm

paths = {
    "hosp_diagnoses_icd": "/data/padmalab_external/special_project/physionet.org/files/mimiciv/3.1/hosp/diagnoses_icd.csv.gz",
    "ed_diagnoses_icd": "/data/padmalab_external/special_project/physionet.org/files/mimic-iv-ed/2.2/ed/diagnosis.csv.gz",
}

hosp_diagnoses_icd = pd.read_csv(paths["hosp_diagnoses_icd"], compression="gzip")
ed_diagnoses_icd   = pd.read_csv(paths["ed_diagnoses_icd"], compression="gzip")

print(hosp_diagnoses_icd.shape)
print(ed_diagnoses_icd.shape)


/home/weijiesun/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(6364488, 5)
(899050, 6)


In [2]:

admissions_path = "/home/shuo19/projects/ecg_preop/data/raw/admissions.csv"
edstays_path    = "/home/shuo19/projects/ecg_preop/data/raw/edstays.csv"

admissions = pd.read_csv(admissions_path)
edstays    = pd.read_csv(edstays_path)

print(admissions.shape)
print(edstays.shape)



(546028, 16)
(425087, 9)


In [3]:
import pandas as pd
ecg_diagnosis_csv = (
    "/data/padmalab_external/special_project/physionet.org/files/"
    "mimic-iv-ecg-ext-icd-labels/1.0.1/records_w_diag_icd10.csv"
)

ecg_diagnoses = pd.read_csv(ecg_diagnosis_csv)
print (ecg_diagnoses[['ed_diag_ed', 'ed_diag_hosp', 'hosp_diag_hosp']].head())

               ed_diag_ed                                       ed_diag_hosp  \
0      ['R4182', 'G9340']  ['F319', 'J449', 'B182', 'E871', 'V462', 'I958...   
1      ['R4182', 'G9340']  ['F319', 'J449', 'B182', 'E871', 'V462', 'I958...   
2                      []                                                 []   
3                      []                                                 []   
4  ['S72092A', 'W1830XA']  ['K219', 'F419', 'Z7901', 'Z87891', 'G43909', ...   

                                      hosp_diag_hosp  
0                                                 []  
1                                                 []  
2  ['J449', 'B182', 'E871', 'R197', 'V462', 'R188...  
3                                                 []  
4                                                 []  


In [4]:
import re


diag_cols = [
    "ed_diag_ed",
    "ed_diag_hosp",
    "hosp_diag_hosp",
    "all_diag_hosp",
    "all_diag_all",
]

def parse_icd_codes(x) -> set[str]:
    """
    Parse a single diagnosis-cell value into a set of clean ICD strings.

    Handles:
      - NaN / None / "" / "[]"
      - strings that look like lists: "['I10','E119']"
      - comma-separated strings: "I10, E119"
      - extra quotes / brackets / whitespace
    """
    if x is None:
        return set()
    if isinstance(x, float) and pd.isna(x):
        return set()

    # Already a list-like
    if isinstance(x, (list, tuple, set)):
        return set(str(v).strip().strip("'").strip('"').upper() for v in x if str(v).strip())

    s = str(x).strip()
    if s == "" or s.lower() == "nan" or s == "[]":
        return set()

    # Remove outer brackets if it's list-like
    s = s.strip()
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        s = s[1:-1].strip()

    if s == "":
        return set()

    # Split on commas
    parts = [p.strip() for p in s.split(",")]

    out = set()
    for p in parts:
        if p == "" or p == "[]":
            continue
        # strip quotes/brackets leftovers
        p = p.strip().strip("'").strip('"').strip()
        p = p.strip("[](){}").strip()
        if p == "" or p == "[]":
            continue
        out.add(p.upper())

    return out


def union_diag_codes_row(row: pd.Series, cols: list[str]) -> list[str]:
    """
    Union codes across diagnosis columns for one row.
    Returns a sorted list (deterministic).
    """
    all_codes = set()
    for c in cols:
        all_codes |= parse_icd_codes(row.get(c))
    return sorted(all_codes)


# Build the union column
ecg_diagnoses["diag_icd_union"] = ecg_diagnoses.apply(
    lambda row: union_diag_codes_row(row, diag_cols),
    axis=1
)



In [ ]:
import re
import yaml
import pandas as pd
from functools import lru_cache


def _snake(name: str) -> str:
    s = str(name).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s


def load_disease_map(yaml_path: str) -> dict:
    with open(yaml_path, "r", encoding="utf-8") as f:
        raw = yaml.safe_load(f) or {}
    out = {}
    for disease, payload in raw.items():
        if not isinstance(payload, dict):
            continue
        icd9 = [str(x).strip().upper() for x in (payload.get("ICD9", []) or []) if str(x).strip()]
        icd10 = [str(x).strip().upper() for x in (payload.get("ICD10", []) or []) if str(x).strip()]
        out[disease] = {"ICD9": icd9, "ICD10": icd10}
    return out


def to_icd_set(x) -> set[str]:
    if x is None:
        return set()
    if isinstance(x, float) and pd.isna(x):
        return set()
    if isinstance(x, (list, tuple, set)):
        return set(str(c).strip().upper() for c in x if str(c).strip())
    s = str(x).strip()
    if s == "" or s == "[]":
        return set()
    parts = re.split(r"[,\s]+", s)
    return set(p.strip().strip("'").strip('"').upper() for p in parts if p.strip() and p.strip() != "[]")


@lru_cache(maxsize=None)
def startswith_any(code: str, prefixes: tuple) -> bool:
    for p in prefixes:
        if code.startswith(p):
            return True
    return False


def match_diseases_by_versions(codes9: set[str], codes10: set[str], disease_map: dict) -> dict:
    res = {}
    for disease, px in disease_map.items():
        p9 = tuple(px.get("ICD9", []))
        p10 = tuple(px.get("ICD10", []))
        hit9 = any(startswith_any(c, p9) for c in codes9) if p9 else False
        hit10 = any(startswith_any(c, p10) for c in codes10) if p10 else False
        res[disease] = 1 if (hit9 or hit10) else 0
    return res


def match_diseases_ecg_episode_both_versions(codes_all: set[str], disease_map: dict) -> dict:
    res = {}
    for disease, px in disease_map.items():
        p9 = tuple(px.get("ICD9", []))
        p10 = tuple(px.get("ICD10", []))
        hit9 = any(startswith_any(c, p9) for c in codes_all) if p9 else False
        hit10 = any(startswith_any(c, p10) for c in codes_all) if p10 else False
        res[disease] = 1 if (hit9 or hit10) else 0
    return res


def collect_ecg_episode_codes_from_union(ecg_row: pd.Series, union_col: str = "diag_icd_union") -> set[str]:
    x = ecg_row.get(union_col)
    if x is None:
        return set()
    if isinstance(x, float) and pd.isna(x):
        return set()
    if isinstance(x, (list, tuple, set)):
        return set(str(c).strip().upper() for c in x if str(c).strip())
    return to_icd_set(x)


def _get_anchor_time(ecg_time, ed_stay_id, hosp_hadm_id, edstays: pd.DataFrame, admissions: pd.DataFrame):
    times = []
    if pd.notna(ecg_time):
        t = pd.to_datetime(ecg_time, errors="coerce")
        if pd.notna(t):
            times.append(t)
    if pd.notna(ed_stay_id):
        m = edstays.loc[edstays["stay_id"] == ed_stay_id, "intime"]
        if len(m) > 0:
            t = pd.to_datetime(m.iloc[0], errors="coerce")
            if pd.notna(t):
                times.append(t)
    if pd.notna(hosp_hadm_id):
        m = admissions.loc[admissions["hadm_id"] == hosp_hadm_id, "admittime"]
        if len(m) > 0:
            t = pd.to_datetime(m.iloc[0], errors="coerce")
            if pd.notna(t):
                times.append(t)
    if not times:
        return pd.NaT
    return min(times)


def fmt_n_pct(n: int, total: int) -> str:
    if total <= 0:
        return f"{n:,} (0.00%)"
    pct = (n / total) * 100.0
    return f"{n:,} ({pct:.2f}%)"


def build_disease_prevalence_table(hx_df: pd.DataFrame, diseases: list[str]) -> pd.DataFrame:
    total = len(hx_df)
    rows = []
    for d in diseases:
        col = f"hx_{_snake(d)}"
        n = int(hx_df[col].sum()) if col in hx_df.columns else 0
        rows.append({"Disease": d, "N (%)": fmt_n_pct(n, total)})
    return pd.DataFrame(rows, columns=["Disease", "N (%)"])


def _prepare_hosp_diag_with_times(hosp_diagnoses_icd: pd.DataFrame, admissions: pd.DataFrame) -> pd.DataFrame:
    adm = admissions[["hadm_id", "admittime", "dischtime"]].copy()
    adm["admittime"] = pd.to_datetime(adm["admittime"], errors="coerce")
    adm["dischtime"] = pd.to_datetime(adm["dischtime"], errors="coerce")

    d = hosp_diagnoses_icd[["subject_id", "hadm_id", "icd_code", "icd_version"]].copy()
    d = d.dropna(subset=["subject_id", "hadm_id", "icd_code", "icd_version"])
    d["icd_code"] = d["icd_code"].astype(str).str.strip().str.upper()

    d = d.merge(adm, on="hadm_id", how="left")
    d["end_time"] = d["dischtime"].fillna(d["admittime"])
    d["end_time"] = pd.to_datetime(d["end_time"], errors="coerce")

    return d


def _prepare_ed_diag_with_times(ed_diagnoses_icd: pd.DataFrame, edstays: pd.DataFrame) -> pd.DataFrame:
    stays = edstays[["stay_id", "intime", "outtime"]].copy()
    stays["intime"] = pd.to_datetime(stays["intime"], errors="coerce")
    stays["outtime"] = pd.to_datetime(stays["outtime"], errors="coerce")

    d = ed_diagnoses_icd[["subject_id", "stay_id", "icd_code", "icd_version"]].copy()
    d = d.dropna(subset=["subject_id", "stay_id", "icd_code", "icd_version"])
    d["icd_code"] = d["icd_code"].astype(str).str.strip().str.upper()

    d = d.merge(stays, on="stay_id", how="left")
    d["end_time"] = d["outtime"].fillna(d["intime"])
    d["end_time"] = pd.to_datetime(d["end_time"], errors="coerce")

    return d


def _build_subject_hosp_code_index(hosp_diag_timed: pd.DataFrame) -> dict:
    idx = {}
    for sid, g in hosp_diag_timed.groupby("subject_id", sort=False):
        cols = ["hadm_id", "end_time", "icd_version", "icd_code"]
        idx[int(sid)] = g[cols].copy()
    return idx


def _build_subject_ed_code_index(ed_diag_timed: pd.DataFrame) -> dict:
    idx = {}
    for sid, g in ed_diag_timed.groupby("subject_id", sort=False):
        cols = ["stay_id", "end_time", "icd_version", "icd_code"]
        idx[int(sid)] = g[cols].copy()
    return idx


def _codes_from_hosp_subject_window(subject_hosp_df: pd.DataFrame, current_hadm_id, window_start, anchor_time) -> tuple[set[str], set[str]]:
    if subject_hosp_df is None or subject_hosp_df.empty:
        return set(), set()

    keep = pd.Series(False, index=subject_hosp_df.index)

    if pd.notna(window_start) and pd.notna(anchor_time):
        et = subject_hosp_df["end_time"]
        keep = keep | ((et >= window_start) & (et <= anchor_time))

    if pd.notna(current_hadm_id):
        keep = keep | (subject_hosp_df["hadm_id"] == current_hadm_id)

    sub = subject_hosp_df.loc[keep]
    if sub.empty:
        return set(), set()

    codes9 = set(sub.loc[sub["icd_version"] == 9, "icd_code"].astype(str))
    codes10 = set(sub.loc[sub["icd_version"] == 10, "icd_code"].astype(str))
    return codes9, codes10


def _codes_from_ed_subject_window(subject_ed_df: pd.DataFrame, current_stay_id, window_start, anchor_time) -> tuple[set[str], set[str]]:
    if subject_ed_df is None or subject_ed_df.empty:
        return set(), set()

    keep = pd.Series(False, index=subject_ed_df.index)

    if pd.notna(window_start) and pd.notna(anchor_time):
        et = subject_ed_df["end_time"]
        keep = keep | ((et >= window_start) & (et <= anchor_time))

    if pd.notna(current_stay_id):
        keep = keep | (subject_ed_df["stay_id"] == current_stay_id)

    sub = subject_ed_df.loc[keep]
    if sub.empty:
        return set(), set()

    codes9 = set(sub.loc[sub["icd_version"] == 9, "icd_code"].astype(str))
    codes10 = set(sub.loc[sub["icd_version"] == 10, "icd_code"].astype(str))
    return codes9, codes10


# def build_ecg_hx_flags(study_ids: list[int]) -> pd.DataFrame:
#     """
#     Fast version.

#     Requires globally available DataFrames:
#       - ecg_diagnoses (must include 'diag_icd_union')
#       - hosp_diagnoses_icd
#       - ed_diagnoses_icd
#       - admissions (hadm_id, admittime, dischtime)
#       - edstays (stay_id, intime, outtime)
#     """
#     

    
#     return hx_df


In [23]:
mimic_hf_df = pd.read_parquet('/data/padmalab_external/special_project/Weijie_Code/MEDs_label/mimic_discharged_hf_readmission.parquet')
mimic_hf_df['ecg_id'] = mimic_hf_df['ecg_id'].astype(int)
study_ids = mimic_hf_df['ecg_id'].unique().tolist()

In [24]:
mimic_hf_df['ecg_id']

0         40689238
1         44458630
2         49036311
3         42709053
4         48339811
            ...   
372828    40643529
372829    46343464
372830    48436051
372831    48683947
372832    41842293
Name: ecg_id, Length: 372833, dtype: int64

In [ ]:
YAML_PATH = "/home/shuo19/projects/ecg_preop/data/raw/icd_codes_map.yaml"
disease_map = load_disease_map(YAML_PATH)


In [13]:
disease_map = {'Heart_failure': disease_map['Heart_failure']}
diseases = list(disease_map.keys())

In [25]:
ecg_sub = ecg_diagnoses.loc[ecg_diagnoses["study_id"].isin(study_ids)].copy()
ecg_sub["ecg_time"] = pd.to_datetime(ecg_sub["ecg_time"], errors="coerce")

# Pre-prepare timed diagnosis tables ONCE
hosp_diag_timed = _prepare_hosp_diag_with_times(hosp_diagnoses_icd, admissions)
ed_diag_timed = _prepare_ed_diag_with_times(ed_diagnoses_icd, edstays)

# Build subject -> diag table index ONCE (fast per-subject slicing)
hosp_by_subject = _build_subject_hosp_code_index(hosp_diag_timed)
ed_by_subject = _build_subject_ed_code_index(ed_diag_timed)

out_cols = ["study_id", "subject_id", "age", "gender"] + [f"hx_{_snake(d)}" for d in diseases]
rows = []

for _, r in ecg_sub.iterrows():
    study_id = int(r["study_id"])
    subject_id = int(r["subject_id"])
    ed_stay_id = r.get("ed_stay_id")
    hosp_hadm_id = r.get("hosp_hadm_id")

    anchor_time = _get_anchor_time(r.get("ecg_time"), ed_stay_id, hosp_hadm_id, edstays, admissions)
    window_start = pd.NaT if pd.isna(anchor_time) else (anchor_time - pd.DateOffset(years=5))

    # ECG episode: match union column against BOTH ICD9 & ICD10 prefixes
    ecg_codes = collect_ecg_episode_codes_from_union(r, union_col="diag_icd_union")
    ecg_hits = match_diseases_ecg_episode_both_versions(ecg_codes, disease_map)

    # Hosp/ED: pull from pre-indexed subject frames
    subj_hosp = hosp_by_subject.get(subject_id)
    subj_ed = ed_by_subject.get(subject_id)

    hosp9, hosp10 = _codes_from_hosp_subject_window(subj_hosp, hosp_hadm_id, window_start, anchor_time)
    ed9, ed10 = _codes_from_ed_subject_window(subj_ed, ed_stay_id, window_start, anchor_time)

    hosp_hits = match_diseases_by_versions(hosp9, hosp10, disease_map)
    ed_hits = match_diseases_by_versions(ed9, ed10, disease_map)

    row_out = {
        "study_id": study_id,
        "subject_id": subject_id,
        "age": r.get("age"),
        "gender": r.get("gender"),
    }

    for d in diseases:
        col = f"hx_{_snake(d)}"
        row_out[col] = 1 if (ecg_hits.get(d, 0) or hosp_hits.get(d, 0) or ed_hits.get(d, 0)) else 0

    rows.append(row_out)

hx_df = pd.DataFrame(rows, columns=out_cols)
hx_df.attrs["disease_prevalence_table"] = build_disease_prevalence_table(hx_df, diseases)

In [32]:
hx_df.rename(columns={"hx_heart_failure": "has_history"}, inplace=True)

In [39]:
mimic_hf_df = mimic_hf_df.merge(hx_df[['study_id', 'has_history']], left_on='ecg_id', right_on='study_id', how='left')
mimic_hf_df.to_parquet('/data/padmalab_external/special_project/Weijie_Code/MEDs_label/mimic_discharged_hf_readmission.parquet', index=False)
mimic_hf_df

,patient_id,ecg_id,ecg_time,discharge_date,target_icd,event_indicator,event_time,censor_time,death_event,study_id,has_history
0,10000032,40689238,2180-07-23 08:44:00,2180-07-25 17:55:00,I50|428,0,2180-09-10 00:00:00,2180-09-10 00:00:00,True,40689238,0
1,10000032,44458630,2180-07-23 09:54:00,2180-07-25 17:55:00,I50|428,0,2180-09-10 00:00:00,2180-09-10 00:00:00,True,44458630,0
2,10000032,49036311,2180-08-06 09:07:00,2180-08-07 17:50:00,I50|428,0,2180-09-10 00:00:00,2180-09-10 00:00:00,True,49036311,0
3,10000285,42709053,2159-11-26 14:29:00,2159-11-26 19:17:00,I50|428,0,2161-11-08 21:06:00,2161-11-08 21:06:00,False,42709053,0
4,10000635,48339811,2136-06-20 08:54:00,2136-06-20 11:30:00,I50|428,0,2143-12-24 12:52:00,2143-12-24 12:52:00,False,48339811,0
...,...,...,...,...,...,...,...,...,...,...,...
372828,19999828,40643529,2147-07-29 11:59:00,2147-08-04 18:10:00,I50|428,0,2149-01-18 17:00:00,2149-01-18 17:00:00,False,40643529,0
372829,19999840,46343464,2164-07-27 08:37:00,2164-07-28 12:15:00,I50|428,0,2164-09-18 00:00:00,2164-09-18 00:00:00,True,46343464,0
372830,19999840,48436051,2164-09-12 05:13:00,2164-09-17 13:42:00,I50|428,0,2164-09-18 00:00:00,2164-09-18 00:00:00,True,48436051,0
372831,19999840,48683947,2164-09-12 12:28:00,2164-09-17 13:42:00,I50|428,0,2164-09-18 00:00:00,2164-09-18 00:00:00,True,48683947,0


Gpt written parsing function 